In [1]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator

c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
c:\Users\DELL\AppData\Local\Programs\Python\Python313\Lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.30.2 at tensorflow/core/framework/

In [2]:
IMG_HEIGHT = 100
IMG_WIDTH = 100
BATCH_SIZE = 32
NUM_CLASSES = 4  # Number of AQI levels

In [3]:
DATASET_DIR = "../Final"  # Replace with your dataset path

In [4]:
# Data augmentation and normalization
train_datagen = ImageDataGenerator(
    rescale=1.0/255,          # Normalize pixel values to [0, 1]
    rotation_range=20,        # Random rotation
    width_shift_range=0.2,    # Horizontal shift
    height_shift_range=0.2,   # Vertical shift
    zoom_range=0.2,           # Random zoom
    horizontal_flip=True,     # Random horizontal flip
    validation_split=0.2      # 20% data for validation
)

In [5]:
# Prepare train and validation datasets
train_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

Found 1451 images belonging to 4 classes.


In [6]:
val_generator = train_datagen.flow_from_directory(
    DATASET_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

Found 361 images belonging to 4 classes.


In [7]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))

In [8]:
base_model.trainable = False

In [9]:
model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),  # Dense layer for learning task-specific features
    Dropout(0.5),                  # Regularization to avoid overfitting
    Dense(NUM_CLASSES, activation='softmax')  # Output layer for classification
])


In [10]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [11]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,782,660 (105.98 MB)

 Trainable params: 4,194,948 (16.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [12]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
)

Epoch 1/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 37s 661ms/step - accuracy: 0.4507 - loss: 2.3558 - val_accuracy: 0.5845 - val_loss: 1.0014
Epoch 2/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 28s 603ms/step - accuracy: 0.5624 - loss: 1.0820 - val_accuracy: 0.5845 - val_loss: 0.9868
Epoch 3/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 34s 737ms/step - accuracy: 0.5844 - loss: 1.0847 - val_accuracy: 0.5845 - val_loss: 0.9876
Epoch 4/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 32s 695ms/step - accuracy: 0.5830 - loss: 1.0794 - val_accuracy: 0.5845 - val_loss: 0.9920
Epoch 5/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 32s 703ms/step - accuracy: 0.5844 - loss: 1.0706 - val_accuracy: 0.5845 - val_loss: 1.0002
Epoch 6/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 30s 658ms/step - accuracy: 0.5844 - loss: 1.0794 - val_accuracy: 0.5845 - val_loss: 1.0400
Epoch 7/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 31s 679ms/step - accuracy: 0.5844 - loss: 1.0991 - val_accuracy: 0.5845 - val_loss: 1.0094
Epoch 8/10
46/46 ━━━━━━━━━━━━━━━━━━━━ 31s 664ms/step - accuracy: 0.5844 - loss: 1.0710 - val_accu

In [15]:
import os

save_path = "aqi_model_augmented.keras"
model.save(save_path)

print("Saved at:", os.path.abspath(save_path))

Saved at: d:\AQI_ESTIMATION\model\augmentation_experiment\aqi_model_augmented.keras


In [17]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np


model = load_model("aqi_model_augmented.keras")


img_path = "../test/Test Image.jpg"


img = image.load_img(img_path, target_size=(100,100))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

class_names = ["Good", "Moderate", "Unhealthy", "Very Unhealthy"]

prediction = model.predict(img_array)
predicted_class = np.argmax(prediction)

print("Predicted AQI class:", class_names[predicted_class])

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicted AQI class: Very Unhealthy


In [18]:
model.input_shape

(None, 100, 100, 3)

In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 4, 4, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,172,558 (137.99 MB)

 Trainable params: 4,194,948 (16.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

 Optimizer params: 8,389,898 (32.00 MB)

In [20]:
train_generator.classes

array([0, 0, 0, ..., 3, 3, 3], shape=(1451,), dtype=int32)

In [21]:
from collections import Counter 
Counter(train_generator.classes)

Counter({np.int32(3): 848,
         np.int32(1): 392,
         np.int32(2): 197,
         np.int32(0): 14})